# Export YOLOv8n-seg `best.pt` to `best.onnx`

Converts the custom-trained segmentation model to ONNX. The exported `best.onnx` is written to the same `weights` folder as `best.pt`.

In [1]:
# Install dependencies (skip if already installed)
%pip install -q ultralytics onnx onnxslim onnxruntime

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from ultralytics import YOLO

# Path to your custom-trained best.pt model
weights_path = Path(r"C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.pt")
assert weights_path.exists(), f"Model not found: {weights_path}"

# Load the model
model = YOLO(str(weights_path))
print(f"Loaded model: {weights_path.name} | task: {model.task}")

Loaded model: best.pt | task: segment


In [3]:
# Export the model to ONNX format
success = model.export(format="onnx", imgsz=640, simplify=True)

print(f"Export successful: {success}")

Ultralytics 8.4.189  Python-3.13.5 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500)


YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs



PyTorch: starting from 'C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 37, 8400), (1, 32, 160, 160)) (6.5 MB)



ONNX: starting export with onnx 1.23.2 opset 18...


ONNX: slimming with onnxslim 0.1.98...


ONNX: export success  2.2s, saved as 'C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx' (12.7 MB)



Export complete (2.4s)
Results saved to C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx
Predict:         yolo predict task=segment model=C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx imgsz=640 
Validate:        yolo val task=segment model=C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx imgsz=640 data=C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\data_clean.yaml 
Visualize:       https://netron.app


Export successful: C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx


In [4]:
# Verify the exported ONNX model loads and inspect its inputs/outputs
import onnxruntime as ort

onnx_path = weights_path.with_suffix(".onnx")
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

print(f"ONNX file: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")
for i in session.get_inputs():
    print(f"Input : {i.name} {i.shape}")
for o in session.get_outputs():
    print(f"Output: {o.name} {o.shape}")

ONNX file: C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx (13.3 MB)
Input : images [1, 3, 640, 640]
Output: output0 [1, 37, 8400]
Output: output1 [1, 32, 160, 160]


In [5]:
# Optional: run a quick prediction with the ONNX model on a test image
test_images = Path(r"C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\test\images")
sample = next(test_images.glob("*.*"), None)

if sample:
    onnx_model = YOLO(str(onnx_path), task="segment")
    results = onnx_model.predict(str(sample), imgsz=640)
    print(f"Ran ONNX inference on {sample.name}: {len(results[0].boxes)} detection(s)")
else:
    print(f"No images found in {test_images}")

Loading C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\seg_benchmark\runs\YOLOv8_yolov8n-seg\weights\best.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.31.0 with CPUExecutionProvider


image 1/1 C:\Users\Derrick Mirindi\Downloads\ID_Card.v3i.yolov8\test\images\61MXomtXn5S-_AC_UF350-350_QL80__jpg.rf.4c0d233c5ae142c125b9e978069549bb.jpg: 640x640 2 person_with_id_cards, 77.1ms


Speed: 4.1ms preprocess, 77.1ms inference, 6.4ms postprocess per image at shape (1, 3, 640, 640)


Ran ONNX inference on 61MXomtXn5S-_AC_UF350-350_QL80__jpg.rf.4c0d233c5ae142c125b9e978069549bb.jpg: 2 detection(s)
